In [1]:
import numpy as np
import cv2
from pyproj import Transformer
from my_vbr_utils.vbr_dataset import vbrInterpolatedDataset, load_calibration, get_paths_from_scene
from my_vbr_utils.utilities import load_scene_correspondences

# --- Config and Dataset Loading ---
# Set these variables to configure your scene and utility path
scene_name = 'ciampino_train0'  # <--- set this
dataset_root_dir = "/datasets/vbr_slam"  # <--- set this
vbr_utils_root = "/home/bjangley/VPR/mast3r-v2/my_vbr_utils"   # <--- set this
calib_path = get_paths_from_scene(dataset_root_dir, scene_name)[-1]
calib = load_calibration(calib_path)
vbr_scene = vbrInterpolatedDataset(dataset_root_dir,scene_name)

vbr_utils_root = '/home/bjangley/VPR/mast3r-v2/my_vbr_utils'
correspondence_json = f"{vbr_utils_root}/GPSalignment/{scene_name}.json"

# Load correspondences
data = load_scene_correspondences(correspondence_json)
image_indices = data['image_indices']
pixel_locations = data['pixel_locations']
locations = data['locations']
gps = data['gps']

Loaded vbrInterpolatedDataset from scene:  ciampino_train0
Images loaded from:  /datasets/vbr_slam/ciampino/ciampino_train0_kitti/camera_left/data
Ground Truth Poses:  /datasets/vbr_slam/ciampino/ciampino_train0/ciampino_train0_gt.txt


## Reproducing Figure 3

In [2]:
import os
import glob
import pandas as pd
import folium
import matplotlib.pyplot as plt
from folium.plugins import PolyLineTextPath

# === CONFIG ===
base_dir = os.path.join(vbr_utils_root, "global_trajectory")
tile_url = "https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}"
save_html = None #os.path.join(base_dir, "all_trajectories.html")     # set None to skip saving
show_direction_arrows = True                                 # set False to disable
# === Layer control (optional) ===
show_layer_toggle = False  # <- set to True if you want checkboxes to toggle trajectories


# === Discover all scene trajectories ===
# scene_csvs = sorted(glob.glob(os.path.join(base_dir, "*", "trajectory.csv")))


# === CONFIGURATION: Select which scenes to include ===
included_scenes = ["ciampino_train0", "ciampino_train1"]  # Modify as needed
# included_scenes=["spagna_train0"]
# === Discover all scene trajectories ===
all_csvs = sorted(glob.glob(os.path.join(base_dir, "*", "trajectory.csv")))
scene_csvs = [
    path for path in all_csvs
    if os.path.basename(os.path.dirname(path)) in included_scenes
]

if not scene_csvs:
    raise FileNotFoundError(f"No matching trajectory.csv files found under {base_dir} for scenes: {included_scenes}")


if not scene_csvs:
    raise FileNotFoundError(f"No trajectory.csv files found under {base_dir}")

# === Load all trajectories ===
scenes = []
all_points = []
for csv_path in scene_csvs:
    scene_name = os.path.basename(os.path.dirname(csv_path))
    df = pd.read_csv(csv_path)
    if {"latitude", "longitude"}.issubset(df.columns) and len(df) > 1:
        coords = list(zip(df["latitude"].values, df["longitude"].values))  # (lat, lon)
        scenes.append((scene_name, coords))
        all_points.extend(coords)

if not scenes:
    raise RuntimeError("No valid trajectories with at least 2 points were found.")

# === Map center ===
center_lat = sum(p[0] for p in all_points) / len(all_points)
center_lon = sum(p[1] for p in all_points) / len(all_points)

m = folium.Map(location=[center_lat, center_lon], zoom_start=16, tiles=None)
folium.TileLayer(tiles=tile_url, attr="Esri World Imagery").add_to(m)

# === Colors per scene ===
custom_colors = [
    # "#00FFFF",  # Cyan
    # "#FFFF00",  # Yellow
    "#FFB6E6",  # Light Pink (more white-pink)
    "#00FF00",  # Neon Green
    # "#FFA500",  # Orange
    # "#FF1493",  # Deep Pink
    # "#1E90FF",  # Dodger Blue
    "#FF0000",  # Bright Red
    "#FFFFFF",  # White (for dark zones)
    "#000000",  # Black (for light zones)
]

# === Add each scene as its own layer ===
legend_entries = []
for i, (scene_name, coords_latlon) in enumerate(scenes):
    color = custom_colors[i]
    fg = folium.FeatureGroup(name=f"{scene_name} ({len(coords_latlon)} pts)", show=True)

    # Polyline expects (lat, lon) → convert to (lat, lon) -> (lat, lon) works
    poly = folium.PolyLine(
        locations=coords_latlon,
        color=color,
        weight=10,            # thicker line
        opacity=0.9
    ).add_to(fg)

    # Optional: add direction arrows along the line
    if show_direction_arrows:
        PolyLineTextPath(
            poly,
            "▶   ",           # arrow glyphs
            repeat=True,
            offset=7,
            attributes={"fill": color, "font-weight": "bold", "font-size": "16"}
        ).add_to(fg)

    # # Label at start
    # start_lat, start_lon = coords_latlon[0]
    # folium.Marker(
    #     location=[start_lat, start_lon],
    #     icon=folium.DivIcon(
    #         html=f"""
    #         <div style="
    #             font-size: 12px; font-weight: 700; color:{color};
    #             background: rgba(255,255,255,0.8); padding: 2px 6px; border-radius: 4px;">
    #             {scene_name} · {len(coords_latlon)} pts
    #         </div>
    #         """
    #     )
    # ).add_to(fg)

    fg.add_to(m)
    legend_entries.append((scene_name, color, len(coords_latlon)))


# === Layer control (optional) ===
if show_layer_toggle:
    folium.LayerControl(collapsed=False).add_to(m)


# === Legend (bottom-left) ===
legend_html = """
<div style="
    position: fixed; bottom: 20px; left: 20px; z-index: 9999;
    background: white; border: 1px solid #999; padding: 10px 12px;
    font-size: 14px; font-weight: 600; max-height: 320px; overflow-y: auto;">
    <div style="margin-bottom:6px;"><b>Trajectories</b></div>
"""
for name, color, cnt in legend_entries:
    legend_html += f"""
    <div style="margin-bottom:4px;">
      <span style="display:inline-block;width:14px;height:14px;background:{color};
                   margin-right:8px;border-radius:2px;"></span>
      {name} &nbsp;
    </div>
    """
legend_html += "</div>"
# m.get_root().html.add_child(folium.Element(legend_html))



# === Save (optional) ===
if save_html:
    m.save(save_html)
    print(f"Saved map -> {save_html}")

# Show in notebook (if you are in Jupyter)
m


In [4]:
scene_name = "ciampino_train0"
global_trajectory=f"{vbr_utils_root}/global_trajectory/{scene_name}/trajectory.csv"
from my_utils.plotting_maps import plot_trajectory_with_heading
import pandas as pd
df = pd.read_csv(global_trajectory)

# Extract lat/lon + headings
aligned_traj = list(zip(df["latitude"], df["longitude"]))
compass_headings = df["heading"].values
# Example usage:
m = plot_trajectory_with_heading(aligned_traj, compass_headings,zoom=16, width="800px", height="650px")
m


subsample_factor = 1

# Subsample aligned_traj and compass_headings
aligned_traj_subsampled = aligned_traj[0:-1:subsample_factor]
compass_headings_subsampled = compass_headings[0:-1:subsample_factor]
m = plot_trajectory_with_heading(aligned_traj_subsampled, compass_headings_subsampled,zoom=16, width="800px", height="650px")
m

interactive(children=(IntSlider(value=0, description='idx', max=30984), Output()), _dom_classes=('widget-inter…

interactive(children=(IntSlider(value=0, description='idx', max=30983), Output()), _dom_classes=('widget-inter…

Map(center=[41.80050837298958, 12.601367534515612], controls=(ZoomControl(options=['position', 'zoom_in_text',…

## Inspecting anchor-query sequences 

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import json
import matplotlib.pyplot as plt
import numpy as np
import ipywidgets as widgets
vbr_scene_traj = vbr_scene.get_local_trajectory()

json_path = f"/home/bjangley/VPR/mast3r-v2/my_vbr_utils/vbr_sequences/{scene_name}.json"
def load_anchor_query_dict(json_file_path):
    """
    Load the anchor-query dictionary from a JSON file.
    """
    with open(json_file_path, 'r') as f:
        loaded_dict = json.load(f)

    # Convert string keys back to tuples
    anchor_query_dict = {tuple(map(int, key.strip("()").split(","))): value for key, value in loaded_dict.items()}
    return anchor_query_dict
anchor_query_dict=load_anchor_query_dict(json_path)
print(anchor_query_dict)


def plot_subplots_with_start_stop(anchor_range):
    query_ranges = anchor_query_dict.get(anchor_range)
    if query_ranges is None:
        fig, ax = plt.subplots(figsize=(8, 4))
        ax.text(0.5, 0.5, f"No query ranges found for anchor range {anchor_range}.", ha='center', va='center')
        ax.axis('off')
        return fig

    n_total = 1 + len(query_ranges)
    n_cols = 3
    n_rows = int(np.ceil(n_total / n_cols))
    fig, axs = plt.subplots(n_rows, n_cols, figsize=(7 * n_cols, 5 * n_rows), squeeze=False)
    axs = axs.flatten()

    seg = vbr_scene_traj
    # 1) Plot the anchor
    anchor_seq = vbr_scene_traj[anchor_range[0]:anchor_range[1]]
    axs[0].plot(seg[:,0], seg[:,1], color='red', alpha=0.3, label="Full Trajectory")
    axs[0].plot(anchor_seq[:,0], anchor_seq[:,1], 'g-', linewidth=2.5, label=f"Anchor")
    # Start marker
    axs[0].plot(anchor_seq[0,0], anchor_seq[0,1], 'o', color='green', markersize=12, label='Start')
    # End marker
    axs[0].plot(anchor_seq[-1,0], anchor_seq[-1,1], 'X', color='darkgreen', markersize=12, label='End')
    axs[0].set_title(f"Anchor Sequence",fontsize=25) #range {anchor_range}
    axs[0].legend(fontsize=15)
    axs[0].grid(True, linestyle='--')

    # 2) Plot each query range
    for i, (qs, qe) in enumerate(query_ranges, start=1):
        query_seq = vbr_scene_traj[qs:qe]
        axs[i].plot(seg[:,0], seg[:,1], color='red', alpha=0.3, label="Full Trajectory")
        axs[i].plot(query_seq[:,0], query_seq[:,1], 'blue', linewidth=2, label=f"Query")
        # Start marker
        axs[i].plot(query_seq[0,0], query_seq[0,1], 'o', color='blue', markersize=12, label='Start')
        # End marker
        axs[i].plot(query_seq[-1,0], query_seq[-1,1], 'X', color='blue', markersize=12, label='End')
        axs[i].set_title(f"Query Sequence",fontsize=25) #range 
        axs[i].legend(fontsize=15)
        axs[i].grid(True, linestyle='--')

    # Hide unused subplots
    for i in range(n_total, len(axs)):
        axs[i].axis('off')

    fig.tight_layout()
    return fig

anchor_ranges = list(anchor_query_dict.keys())
range_names = [str(r) for r in anchor_ranges]
range_selector = widgets.SelectionSlider(
    options=range_names,
    description='Anchor:',
    continuous_update=False
)

def update_plot(anchor_range_str):
    anchor_range = eval(anchor_range_str)
    plt.close('all')
    return plot_subplots_with_start_stop(anchor_range)

widget = widgets.interactive(update_plot, anchor_range_str=range_selector)
display(widget)

{(0, 1145): [[3026, 3421]], (1145, 1496): [[3421, 3772], [16708, 17244]], (1496, 2456): [[29980, 30849]], (3773, 4594): [[17246, 18245]], (4600, 5390): [[6000, 7300], [18300, 19000]], (13236, 14000): [[11470, 12115]], (14396, 15046): [[24900, 26000]], (15113, 16612): [[21360, 21900], [26986, 27735]]}


interactive(children=(SelectionSlider(continuous_update=False, description='Anchor:', options=('(0, 1145)', '(…